# Overfitting, Underfitting & Model Diagnosis - Assignment

Welcome to the assignment for Model Diagnosis.

One of the most critical skills in machine learning is the ability to diagnose why a model is not performing as well as expected. Is it too simple to capture the underlying patterns (underfitting)? Or is it merely memorizing the training data (overfitting)? Understanding the bias-variance tradeoff is key to answering these questions.

In this assignment, you will explore various diagnostic tools and techniques to identify and fix model problems. You will work with learning curves to visualize model performance as data size increases, analyze the effect of model complexity on bias and variance, and apply regularization to control overfitting. You will also build a robust framework for comparing models and implement a learning curve generator from scratch.

Correctly diagnosing model issues saves time by directing your efforts effectively—knowing whether to collect more data, add features, increase model complexity, or apply regularization.

**What You Will Do in This Assignment**

*   Plot and interpret **Learning Curves** to diagnose bias and variance.
*   Demonstrate the **Bias-Variance Tradeoff** using polynomial regression.
*   Apply **Regularization** (Ridge/Lasso) to control model complexity.
*   Analyze **Cross-Validation Variance** to identify unstable models.
*   Build a systematic **Model Comparison Framework** with statistical tests.
*   Implement a **Learning Curve Generator** from scratch.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

*   All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

*   In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

*   You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

*   Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

*   To submit your notebook for grading, first save it by clicking the save icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1.  [Introduction to Model Diagnosis](#1)
2.  [Learning Curves](#2)
    *   [Exercise 1: Plot and Diagnose Learning Curves](#ex-1)
3.  [Bias-Variance Tradeoff](#3)
    *   [Exercise 2: Polynomial Regression Analysis](#ex-2)
4.  [Complexity Control](#4)
    *   [Exercise 3: Regularization for Overfitting](#ex-3)
5.  [Cross-Validation Variance](#5)
    *   [Exercise 4: Analyze CV Stability](#ex-4)
6.  [Model Comparison Framework](#6)
    *   [Exercise 5: Systematic Model Comparison](#ex-5)
7.  [From-Scratch Implementation](#7)
    *   [Exercise 6: Learning Curve Generator](#ex-6)
8.  [Conclusion](#8)

<a name='1'></a>
## 1 - Introduction to Model Diagnosis

**Model Diagnosis** involves using quantitative and visual tools to understand model performance characteristics.

### Key Concepts:

**Bias (Underfitting)**:
*   Error due to overly simplistic assumptions.
*   Model cannot capture underlying patterns.
*   Symptoms: High training error, high validation error.

**Variance (Overfitting)**:
*   Error due to sensitivity to fluctuations in the training set.
*   Model captures noise as signal.
*   Symptoms: Low training error, high validation error (large gap).

**Bias-Variance Tradeoff**:
*   Increasing complexity $\rightarrow$ Lower Bias, Higher Variance.
*   Decreasing complexity $\rightarrow$ Higher Bias, Lower Variance.
*   Goal: Find the "sweet spot" with low total error.

### Diagnostic Tools:

1.  **Learning Curves**: Performance vs. Training Set Size.
2.  **Validation Curves**: Performance vs. Hyperparameter (Complexity).
3.  **Cross-Validation**: Robust performance estimation.
4.  **Residual Analysis**: Analyzing prediction errors.

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.model_selection import learning_curve, validation_curve, train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.datasets import make_regression, load_digits
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")

### Helper Functions

In [ ]:
def plot_curves(train_sizes, train_scores, val_scores, title="Learning Curve"):
    """
    Generic plotting function for learning curves.
    """
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training score")
    plt.plot(train_sizes, val_mean, 'o-', color="g", label="Cross-validation score")

    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color="g")

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    plt.legend(loc="best")
    plt.grid(True, alpha=0.3)
    plt.show()

print("Helper functions defined!")

### Load Dataset
We will use a synthetic regression dataset for clear visualization of bias-variance concepts, and the digits dataset for classification tasks.

In [ ]:
# Generate synthetic data for regression
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)
X_reg = X_reg ** 2 + 3 # Add non-linearity

# Load digits dataset for classification
digits = load_digits()
X_cls, y_cls = digits.data, digits.target

print(f"Regression Data: {X_reg.shape}")
print(f"Classification Data: {X_cls.shape}")

<a name='2'></a>
## 2 - Learning Curves

**Learning Curves** plot the model's performance on the training set and the validation set as a function of the training set size. They are widely used to diagnose whether a learning algorithm suffers from high bias, high variance, or both.

### Interpretation:

*   **High Bias (Underfitting)**:
    *   Training error and validation error converge to a similar value.
    *   Usually, the error is high (score is low).
    *   Adding more data is unlikely to help.

*   **High Variance (Overfitting)**:
    *   Large gap between training error and validation error.
    *   Training error is low (score is high).
    *   Validation error is high (score is low).
    *   Adding more data is likely to help.

<a name='ex-1'></a>
### Exercise 1 - Plot and Diagnose Learning Curves

**Your Task:**

Implement the `generate_learning_curve` function that:
1.  Uses `learning_curve` from sklearn.
2.  Computes training and test scores for varying training set sizes.
3.  Returns the sizes, train scores, and test scores.

<details>
  <summary><b><font color="green">Code Hints</font></b></summary>

*   Use: `train_sizes, train_scores, test_scores = learning_curve(...)`
*   Pass the `estimator`, `X`, `y`.
*   Set `cv=kf` (cross-validation splitter).
*   Set `train_sizes=np.linspace(0.1, 1.0, 10)` to check 10 points.
*   Use `n_jobs=-1` for parallel processing.
</details>

In [ ]:
# GRADED FUNCTION: generate_learning_curve
def generate_learning_curve(estimator, X, y, cv=5):
    """
    Generate learning curve data.

    Args:
        estimator: The model to evaluate
        X: Features
        y: Target
        cv: Cross-validation generator or int

    Returns:
        dict containing train_sizes, train_mean, val_mean
    """
    ### START CODE HERE ### (≈ 5-8 lines)
    
    # Generate array of training set sizes (from 10% to 100%)
    train_sizes_seq = None

    # Compute learning curve using sklearn's learning_curve
    # Hint: scoring='neg_mean_squared_error', n_jobs=-1
    train_sizes, train_scores, val_scores = None
    
    # Calculate means (convert from neg_mean_squared_error to RMSE)
    # Hint: RMSE = sqrt(-mean_score)
    train_mean = None
    val_mean = None
    
    ### END CODE HERE ###
    
    return {
        'train_sizes': train_sizes,
        'train_mean': train_mean,
        'val_mean': val_mean,
        'train_scores': train_scores,
        'val_scores': val_scores
    }

In [ ]:
# Test with Linear Regression (Likely Underfitting)
lr_model = LinearRegression()
curve_data_lr = generate_learning_curve(lr_model, X_reg, y_reg)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(curve_data_lr['train_sizes'], curve_data_lr['train_mean'], 'o-', label='Training Error (RMSE)')
plt.plot(curve_data_lr['train_sizes'], curve_data_lr['val_mean'], 'o-', label='Validation Error (RMSE)')
plt.title('Learning Curve: Linear Regression (Simple Model)', fontsize=14)
plt.xlabel('Training Set Size')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True)
plt.show()

# You should see errors converging to a relatively high value (~30-40), indicating underfitting.

In [ ]:
unittests.exercise_1(generate_learning_curve)

<a name='3'></a>
## 3 - Bias-Variance Tradeoff

The tradeoff between bias and variance is controlled by model complexity.
*   **Low Complexity**: High Bias, Low Variance.
*   **High Complexity**: Low Bias, High Variance.

We can visualize this using Polynomial Regression by varying the degree of the polynomial.

<a name='ex-2'></a>
### Exercise 2 - Polynomial Regression Analysis

**Your Task:**

Implement `evaluate_poly_degrees` that:
1.  Iterates through a list of polynomial degrees.
2.  Creates a Pipeline with `PolynomialFeatures` + `LinearRegression`.
3.  Evaluates MSE using cross-validation.
4.  Returns results for comparison.

<details>
  <summary><b><font color="green">Code Hints</font></b></summary>

*   Loop: `for degree in degrees:`
*   Pipeline: `model = make_pipeline(PolynomialFeatures(degree), LinearRegression())`
*   Score: `scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')`
*   RMSE: `np.sqrt(-np.mean(scores))`
</details>

In [ ]:
# GRADED FUNCTION: evaluate_poly_degrees
def evaluate_poly_degrees(X, y, degrees):
    """
    Evaluate polynomial regression models with different degrees.

    Args:
        X, y: Dataset
        degrees: List of degrees to test

    Returns:
        List of dictionaries with degree and RMSE
    """
    results = []
    
    ### START CODE HERE ### (≈ 10 lines)
    for degree in degrees:
        # Create pipeline: PolynomialFeatures -> LinearRegression
        model = None
        
        # Calculate CV scores (use neg_mean_squared_error)
        scores = None
        
        # Convert to RMSE (root mean squared error)
        rmse = None
        
        # Append to results
        results.append({
            'degree': degree,
            'rmse': rmse
        })
    ### END CODE HERE ###
    
    return results

In [ ]:
# Evaluate degrees
degrees = [1, 2, 3, 5, 10, 15]
poly_results = evaluate_poly_degrees(X_reg, y_reg, degrees)

# Convert to DF
df_poly = pd.DataFrame(poly_results)
print(df_poly)

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(df_poly['degree'], df_poly['rmse'], 'o-', linewidth=2)
plt.title('RMSE vs Polynomial Degree', fontsize=14)
plt.xlabel('Degree (Complexity)')
plt.ylabel('CV RMSE')
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_2(evaluate_poly_degrees)

<a name='4'></a>
## 4 - Complexity Control

When a model is overfitting (High Variance), **Regularization** is a key technique to fix it. It penalizes complex models, effectively simplifying them.

*   **Ridge (L2)**: Adds penalty $\alpha \sum w_i^2$.
*   **Lasso (L1)**: Adds penalty $\alpha \sum |w_i|$.

<a name='ex-3'></a>
### Exercise 3 - Regularization for Overfitting

**Your Task:**

Implement `test_regularization` that:
1.  Takes a high-degree polynomial (e.g., degree 15) which overfits.
2.  Applies Ridge regression with different alpha values.
3.  Returns scores for each alpha.

<details>
  <summary><b><font color="green">Code Hints</font></b></summary>

*   Create base features: `poly = PolynomialFeatures(degree=15)`
*   Loop alphas: `for alpha in alphas:`
*   Model: `Ridge(alpha=alpha)` (remember to scale data!)
*   Use Pipeline: `make_pipeline(poly, StandardScaler(), ridge)`
</details>

In [ ]:
# GRADED FUNCTION: test_regularization
def test_regularization(X, y, alphas):
    """
    Test Ridge regularization strength on a high-degree polynomial.

    Args:
        X, y: Dataset
        alphas: List of alpha values

    Returns:
        List of results
    """
    results = []
    
    ### START CODE HERE ### (≈ 10-12 lines)
    for alpha in alphas:
        # Create pipeline: Poly -> Scaling -> Ridge
        # Hint: Use make_pipeline with PolynomialFeatures(degree=15), StandardScaler(), and Ridge(...)
        model = None
        
        # Calculate CV Score (neg_mean_squared_error)
        scores = None
        
        # Calculate RMSE
        rmse = None
        
        # Append results
        results.append({
            'alpha': alpha,
            'rmse': rmse
        })
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test alphas
alphas = [0.0001, 0.01, 1, 10, 100, 1000]
reg_results = test_regularization(X_reg, y_reg, alphas)
df_reg = pd.DataFrame(reg_results)

print(df_reg)

plt.figure(figsize=(10, 6))
plt.semilogx(df_reg['alpha'], df_reg['rmse'], 'o-', color='purple')
plt.title('Effect of Regularization (Ridge) on RMSE', fontsize=14)
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('RMSE')
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_3(test_regularization)

<a name='5'></a>
## 5 - Cross-Validation Variance

Sometimes a model has a good *average* score, but is **unstable**—it performs very well on some folds and very poorly on others. This indicates the model is sensitive to the specific data points in the splits.

We can analyze the **standard deviation** of cross-validation scores.

<a name='ex-4'></a>
### Exercise 4 - Analyze CV Stability

**Your Task:**

Implement `analyze_cv_stability`:
1.  Run cross-validation.
2.  Calculate mean score AND standard deviation.
3.  Return both.

<details>
  <summary><b><font color="green">Code Hints</font></b></summary>
  
*   `scores = cross_val_score(...)`
*   `mean = scores.mean()`
*   `std = scores.std()`
</details>

In [ ]:
# GRADED FUNCTION: analyze_cv_stability
def analyze_cv_stability(model, X, y, cv=10):
    """
    Analyze stability of a model using CV variance.

    Args:
        model: Sklearn model
        X, y: Data
        cv: Folds

    Returns:
        mean, std, scores
    """
    ### START CODE HERE ### (≈ 3 lines)
    
    # Run cross-validation
    scores = None
    
    # Calculate mean and standard deviation
    mean_score = None
    std_score = None
    
    ### END CODE HERE ###
    
    return mean_score, std_score, scores

In [ ]:
# Compare Random Forest vs Decision Tree on Digits
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)

mean_dt, std_dt, scores_dt = analyze_cv_stability(dt, X_cls, y_cls)
mean_rf, std_rf, scores_rf = analyze_cv_stability(rf, X_cls, y_cls)

print(f"Decision Tree: {mean_dt:.4f} (+/- {std_dt:.4f})")
print(f"Random Forest: {mean_rf:.4f} (+/- {std_rf:.4f})")

plt.figure(figsize=(10, 6))
plt.boxplot([scores_dt, scores_rf], labels=['Decision Tree', 'Random Forest'])
plt.title('Model Stability Comparison', fontsize=14)
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_4(analyze_cv_stability)

<a name='6'></a>
## 6 - Model Comparison Framework

Comparing models systematically requires looking at multiple metrics and ensuring differences are statistically significant.

<a name='ex-5'></a>
### Exercise 5 - Systematic Model Comparison

**Your Task:**

Implement `compare_models`:
1.  Take a dictionary of models.
2.  Evaluate each using CV.
3.  Perform a t-test against a baseline (first model).
4.  Return DataFrame with Mean, Std, and P-value.

<details>
  <summary><b><font color="green">Code Hints</font></b></summary>
  
*   Store scores for all models in a dict `all_scores`.
*   Baseline scores: `baseline_scores = all_scores[baseline_name]`.
*   T-test: `t_stat, p_val = stats.ttest_rel(baseline_scores, current_scores)` (using related/paired samples).
</details>

In [ ]:
# GRADED FUNCTION: compare_models
def compare_models(models_dict, X, y, cv=10):
    """
    Systematically compare models with statistical tests.

    Args:
        models_dict: Dict of {name: model}
        X, y: Data

    Returns:
        DataFrame with results
    """
    results = []
    
    ### START CODE HERE ### (≈ 15-20 lines)
    
    # 1. Run CV for all models
    all_scores = {} # Dictionary to store scores for each model
    model_names = list(models_dict.keys())
    baseline_name = model_names[0]
    
    # Iterate through models and get CV scores
    for name, model in models_dict.items():
        all_scores[name] = None 
        
    # 2. Compare against baseline
    baseline_scores = all_scores[baseline_name]
    
    for name in model_names:
        current_scores = None # Get scores for current model
        mean = None
        std = None
        
        # Calculate p-value (paired t-test)
        if name == baseline_name:
            p_value = 1.0
        else:
            # Use stats.ttest_rel for paired samples
            _, p_value = None
            
        results.append({
            'Model': name,
            'Mean Accuracy': mean,
            'Std Dev': std,
            'P-Value (vs Baseline)': p_value
        })
        
    ### END CODE HERE ###
    
    return pd.DataFrame(results)

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

comparison_results = compare_models(models, X_cls, y_cls)
print(comparison_results)

In [ ]:
unittests.exercise_5(compare_models)

<a name='7'></a>
## 7 - From-Scratch Implementation

Understanding how learning curves are built helps demystify the process.

<a name='ex-6'></a>
### Exercise 6 - Learning Curve Generator

**Your Task:**

Implement `learning_curve_scratch` that:
1.  Takes a model, X, y.
2.  Loops through specified training sample sizes.
3.  In each loop, manually splits data (e.g., using `train_test_split`).
4.  Trains on the subset, evaluates on the *full* validation set (or held-out set).
5.  Recommended: Repeat K times per size to get stable estimates (Monte Carlo styles).

For simplicity, let's do a single split validation, but growing the training set.

<details>
  <summary><b><font color="green">Code Hints</font></b></summary>
  
*   Set aside test set *once* at start: `X_train_full, X_test, y_train_full, y_test = train_test_split(...)`
*   Loop fractions `[0.1, ... 1.0]`.
*   Subsample: `num_samples = int(frac * len(X_train_full))`
*   Slice: `X_subset = X_train_full[:num_samples]`
*   Train on subset, Score on subset (Train Err), Score on `X_test` (Val Err).
</details>

In [ ]:
# GRADED FUNCTION: learning_curve_scratch
def learning_curve_scratch(estimator, X, y, train_fractions=np.linspace(0.1, 1.0, 10)):
    """
    Generate learning curve data from scratch.

    Args:
        estimator: Model
        X, y: Data
        train_fractions: Fractions of training data to use

    Returns:
        sizes, train_scores, val_scores
    """
    
    ### START CODE HERE ### (≈ 15 lines)
    
    # 1. Hold out a validation set using train_test_split (20% is good)
    X_train_full, X_val, y_train_full, y_val = None, None, None, None
    
    sizes = []
    train_scores = []
    val_scores = []
    
    n_train_total = len(X_train_full)
    
    for frac in train_fractions:
        # Determine subset size (based on fraction)
        n_subset = None
        if n_subset == 0: continue
        
        # Create subset (slice the full training set)
        X_sub = None
        y_sub = None
        
        # Train model on subset. Hint: use sklearn.base.clone to get a fresh model
        # model = clone(estimator)
        # model.fit(...)
        
        # Evaluate on subset (Train score) and validation set (Validation score)
        train_sc = None
        val_sc = None
        
        sizes.append(n_subset)
        train_scores.append(train_sc)
        val_scores.append(val_sc)

    ### END CODE HERE ###
    
    return sizes, train_scores, val_scores

In [ ]:
# Test from scratch implementation
rf_scratch = RandomForestClassifier(random_state=42, n_estimators=10)
sizes, t_sc, v_sc = learning_curve_scratch(rf_scratch, X_cls, y_cls)

plt.figure(figsize=(10, 6))
plt.plot(sizes, t_sc, 'o-', label='Train Accuracy')
plt.plot(sizes, v_sc, 'o-', label='Validation Accuracy')
plt.title('Learning Curve (From Scratch)', fontsize=14)
plt.xlabel('Training Examples')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
unittests.exercise_6(learning_curve_scratch)

<a name='8'></a>
## 8 - Conclusion

**Congratulations!** You have completed the Model Diagnosis assignment.

### Key Takeaways:

1.  **Learning Curves** are your first line of defense.
    *   Converging at low score $\rightarrow$ **Bias** (Underfitting).
    *   Large gap $\rightarrow$ **Variance** (Overfitting).
2.  **Complexity Control**:
    *   Simple models (Linear) need more features/complexity to fix bias.
    *   Complex models (High-deg Polynomial, Deep Trees) need regularization/more data to fix variance.
3.  **Stability Matters**:
    *   Always check the standard deviation of CV scores. A "lucky" split can mislead you.
4.  **Rigorous Comparison**:
    *   Use statistical tests (like t-test) to confirm if Model A is *really* better than Model B.

Good luck with your future model diagnostics!